# Iris Classification — R

Machine learning workflow for the Iris dataset:
1. Load and summarize the data
2. Split into 80% training and 20% validation
3. Train multiple algorithms and evaluate on the validation set
4. Report the best-performing model

## Import libraries

In [ ]:
library(caret)

## Load dataset

The CSV has no header row. Column names follow the standard Iris attribute names.

In [ ]:
url <- "https://raw.githubusercontent.com/jbrownlee/Datasets/master/iris.csv"
dataset <- read.csv(url, header = FALSE)
colnames(dataset) <- c("Sepal.Length", "Sepal.Width", "Petal.Length", "Petal.Width", "Species")
dataset$Species <- as.factor(dataset$Species)
cat("Loaded", nrow(dataset), "rows and", ncol(dataset), "columns\n")

## Summarize the dataset

In [ ]:
cat("Dimensions:\n")
print(dim(dataset))
cat("\nFirst 10 rows:\n")
print(head(dataset, 10))
cat("\nStatistical summary of numeric columns:\n")
print(summary(dataset[, 1:4]))
cat("\nClass distribution:\n")
print(table(dataset$Species))

## Create validation dataset

Reserve 20% of the data for validation; the remaining 80% is used for training.

In [ ]:
set.seed(1)
train_index <- createDataPartition(dataset$Species, p = 0.80, list = FALSE)
training <- dataset[train_index, ]
validation <- dataset[-train_index, ]

cat("Training set:", nrow(training), "samples (", round(nrow(training) / nrow(dataset) * 100), "%)\n", sep = "")
cat("Validation set:", nrow(validation), "samples (", round(nrow(validation) / nrow(dataset) * 100), "%)\n", sep = "")

## Train and compare algorithms

Each model is fit on the training set and evaluated on the held-out validation set.

In [ ]:
control <- trainControl(method = "none")
metric <- "Accuracy"

set.seed(1)
fit.lda <- train(Species ~ ., data = training, method = "lda", metric = metric, trControl = control)
set.seed(1)
fit.cart <- train(Species ~ ., data = training, method = "rpart", metric = metric, trControl = control)
set.seed(1)
fit.knn <- train(Species ~ ., data = training, method = "knn", metric = metric, trControl = control)
set.seed(1)
fit.svm <- train(Species ~ ., data = training, method = "svmRadial", metric = metric, trControl = control)
set.seed(1)
fit.rf <- train(Species ~ ., data = training, method = "rf", metric = metric, trControl = control)

models <- list(
  "Linear Discriminant Analysis" = fit.lda,
  "Classification and Regression Tree" = fit.cart,
  "k-Nearest Neighbors" = fit.knn,
  "Support Vector Machine" = fit.svm,
  "Random Forest" = fit.rf
)

results <- data.frame(
  algorithm = character(),
  accuracy = numeric(),
  stringsAsFactors = FALSE
)

for (name in names(models)) {
  predictions <- predict(models[[name]], validation)
  accuracy <- mean(predictions == validation$Species)
  results <- rbind(results, data.frame(algorithm = name, accuracy = accuracy))
  cat(name, ":", sprintf("%.4f", accuracy), "\n")
}

## Best model

In [ ]:
best_row <- results[which.max(results$accuracy), ]
cat("Best algorithm:", best_row$algorithm, "\n")
cat("Validation accuracy:", sprintf("%.4f", best_row$accuracy), "\n")